# AtCoder Beginner Contest 477

- https://atcoder.jp/contests/abc477

`-` 조금 늦게 시작했음. 오늘이 툐요일인줄 몰랐다

`-` 핸드폰 알람이 울리길래 봤더니 앳코더라고 메모되어 있었다. 뭐? 오늘 토요일? 바로 참여해버렸는데 하길 잘했다 (이거 안 했다고 어차피 뭐 안 했을 것임 + 5솔)

`-` 풀이 시간은 오래 걸렸지만 D, E번 문제와 비슷한 유형을 풀어봐서 어렵지 않았다. 그냥 오래만 걸렸을 뿐이다 (진짜)

`-` 아는 거랑 비슷하다고 해도 너무 패턴 매칭으로 접근하면 안 된다 (논리 학습이 안 됨). 진짜 깡 웰노운 아니고서야 논리를 설계하고 패턴 매칭으로 덮는 느낌으로 가자 

## A - Traffic Light (3:49)

`-` 브론즈 $5$

In [62]:
def solution():
    c = input().rstrip()
    if c == "B":
        answer = "Y"
    elif c == "Y":
        answer = "R"
    else:
        answer = "B"
    print(answer)


solution()

# input
# B

 B


Y


## B - Standing Outliers (7:04)

`-` 브루트 포스로 $O\left(N^2\right)$에 계산할 수 있다

In [3]:
def solution():
    N, D = map(int, input().split())
    X = list(map(int, input().split()))
    answer = []
    for i, x_i in enumerate(X):
        stands_apart = all(abs(x_i - X[j]) >= D for j in range(N) if i != j) 
        if stands_apart:
            answer.append(i + 1)
    print(len(answer))
    print(*answer)


solution()

# input
# 3 3
# 1 5 7

 3 3
 1 5 7


1
1


## C - Range Search Query (25:59)

`-` $S[L \dots R]$에 $T$가 포함되는지 판단해야 한다. $S[i \dots i + \operatorname{len}(T) - 1] = T$를 만족하는 $i$를 오름차순으로 정렬해서 가지고 있자

`-` 쿼리가 들어오면 이분 탐색으로 $L$보다 크거나 같은 $i$ 중 최솟값을 찾자. $i + len(T) - 1 \le R$이면 부분 문자열에 $T$가 포함된다

`-` 전체 알고리즘의 시간 복잡도는 최악의 경우 $O(|S|\cdot |T| + Q\log\frac{|S|}{|T|})$이다

`-` 만약 조건을 만족하는 $i$를 KMP 알고리즘으로 찾는다면 시간 복잡도를 최악의 경우에도 $O(|S| + |T| + Q\log\frac{|S|}{|T|})$로 줄일 수 있다

In [4]:
from bisect import bisect_right


def solution():
    Q = int(input())
    S = input().rstrip()
    T = input().rstrip()
    n, m = len(S), len(T)
    indices = [i for i in range(n - m + 1) if S[i : i + m] == T]
    for _ in range(Q):
        L, R = map(lambda x: int(x) - 1, input().split())
        if not indices or indices[-1] < L:
            print("No")
            continue
        rank = bisect_right(indices, L - 1)
        if indices[rank] + m - 1 <= R:
            print("Yes")
        else:
            print("No")       


solution()

# input
# 3
# abcdabc
# bc
# 2 6
# 3 5
# 3 7

 3
 abcdabc
 bc
 2 6


Yes


 3 5


No


 3 7


Yes


`-` 인덱스 처리 하는 데 은근 오래 걸림. 미리 정리하고 구현하자

## D - Masking Tape (48:16)

`-` 나이브하게 처리하면 $2$번 쿼리만 들어오는 최악의 경우에 $O(NQ)$로 TLE이다

`-` 타일이 없는 정사각형 입장에선 가장 최근에 입력된 $2$번 쿼리만 유의미하다. 정사각형마다 타일 유무와 가장 최근에 들어온 $1$번 쿼리 순번을 저장하자. 그리고 $1$번 쿼리가 입력될 때 $2$번 쿼리에 의한 색깔 변화를 반영할 것이다

`-` 만약 타일이 있다면 색깔 변화는 없다. 반대로 타일이 없다면 가장 최근에 입력된 $1$번 쿼리 순번과 $2$번 쿼리 순번을 비교하자. 만약 $2$번 쿼리 순번이 나중이라면 $1$번 쿼리에 의해 타일이 없어진 뒤에 $2$번 쿼리가 입력된 것이니 그때의 색깔로 변경해야 한다. 그렇지 않다면 무시하자

`-` 모든 쿼리가 입력된 후 마지막 $2$번 쿼리를 반영하고자 각 정사각형마다 강제 갱신(가상의 $1$번 쿼리가 입력됐다고 생각)을 해주자. 각 쿼리마다 $O(1)$에 처리 가능하므로 전체 알고리즘의 시간 복잡도는 $O(N + Q)$이다

In [13]:
def solution():
    N, Q = map(int, input().split())
    colors = ["a"] * N
    has_tile_mask = [False] * N
    last_query1_indices = [-1] * N
    last_query2_color = "a"
    last_query2_index = -1
    for i in range(Q):
        query = input().split()
        if query[0] == "2":
            C = query[1]
            last_query2_color = C
            last_query2_index = i
        else:
            X = int(query[1]) - 1
            if not has_tile_mask[X] and last_query1_indices[X] < last_query2_index:
                colors[X] = last_query2_color
            has_tile_mask[X] ^= True
            last_query1_indices[X] = i
    for x in range(N):
        if not has_tile_mask[x] and last_query1_indices[x] < last_query2_index:
            colors[x] = last_query2_color
    print("".join(colors))


solution()

# input
# 5 5
# 1 1
# 2 c
# 1 1
# 1 1
# 1 1

 5 5
 1 1
 2 c
 1 1
 1 1
 1 1


acccc


`-` 반례 안 만들고 예제만 맞히면 바로 제출했는데 너무 많이 틀려버렸다. 틀릴 때마다 깔짝깔짝 고쳐서 계속 틀림;

`-` 반례 좀 미리미리 만들자. 남탓을 하자면 예제가 너무 쉽다 

## E - 	Wheel Distance (79:33)

`-` $1 \leftrightarrow 2 \leftrightarrow \cdots \leftrightarrow N \leftrightarrow 1$ 형식의 무방향 원 그래프가 주어진다. 추가로 각 정점은 $N+1$번 정점과 연결되어 있다. 이는 워프라고 부르겠다

`-` 임의의 최단 경로에 워프가 $2$번 이상 포함될 수 없다. 왜냐하면 첫 번째 워프의 출발점에서 두 번째 워프의 도착점으로 $1$번만에 워프하면 항상 거리가 더 작기 때문이다

`-` 따라서 $S$에서 $T$로 가는 최단 경로엔 워프가 없거나 $1$번이거나 둘 중 하나이다. 만약 워프가 없다면 $S \to \cdots\to T$가 시계 방향이거나 반시계 방향이거나 둘 중 하나이다. 이를 $O(1)$에 계산하기 위해 누적 합 배열을 전처리 해두자

`-` 워프가 있다면 $S \to \cdots \to N + 1 \to \cdots \to T$이다. 이때 무방향 그래프이므로 이때의 최단 거리는 $N + 1$에서 $S$로 가는 최단 거리와 $T$로 가는 최단 거리의 합과 같다. 이 역시 $N+1$번 정점을 source로 하는 데이크스트라를 $O(N \log N)$에 전처리 해두면 $O(1)$에 구할 수 있다

`-` 따라서 전체 알고리즘의 시간 복잡도는 $O(N \log N  + Q)$이다. 단, 쿼리에 $N+1$이 포함된 경우 무조건 워프를 사용해야 함에 주의하자

In [6]:
import heapq
from itertools import accumulate


def dijkstra(graph, source):
    inf = 10**18
    n = len(graph)
    distances = [inf] * n
    distances[source] = 0
    Q = [(0, source)]
    while Q:
        d_u, u = heapq.heappop(Q)
        if distances[u] < d_u:
            continue
        for v, w in graph[u]:
            d_v = d_u + w
            if d_v < distances[v]:
                distances[v] = d_v
                heapq.heappush(Q, (d_v, v))
    return distances


def solution():
    N, Q = map(int, input().split())
    A = list(map(int, input().split()))
    B = list(map(int, input().split()))
    graph = [[] for _ in range(N + 2)]
    for i in range(1, N + 1):
        j = (i % N) + 1
        w_a, w_b = A[i - 1], B[i - 1]
        graph[i].append((j, w_a))
        graph[j].append((i, w_a))
        graph[i].append((N + 1, w_b))
        graph[N + 1].append((i, w_b))
    distances = dijkstra(graph, N + 1)
    Ap = [*accumulate(A, initial=0)]
    total = Ap[-1]
    for _ in range(Q):
        S, T = map(int, input().split())
        if S > T:
            S, T = T, S
        d1 = distances[S] + distances[T]
        if T == N + 1:
            print(d1)
            continue
        d2 =  Ap[T - 1] - Ap[S - 1]
        d3 = total - d2
        answer = min(d1, d2, d3)
        print(answer)


solution()

# input
# 5 3
# 1 3 4 2 5
# 55 55 55 55 55
# 2 5
# 5 6
# 1 4

 5 3
 1 3 4 2 5
 55 55 55 55 55
 2 5


6


 5 6


55


 1 4


7
